In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Sources ──────────────────────────────────────────────────────────
silver = spark.table("iran_israel_capstone_project.silver.daily_market_clean")
event_dim = spark.table("iran_israel_capstone_project.silver.event_dim")
cpi_raw = spark.table("iran_israel_capstone_project.bronze.macro_cpi_raw")

# ════════════════════════════════════════════════════════════════════
# STEP 1: Monthly market aggregation from daily silver data
# ════════════════════════════════════════════════════════════════════
w_month = Window.partitionBy("year_month").orderBy(F.col("trade_date").desc())

daily = (
    silver
    .withColumn("year_month", F.date_format("trade_date", "yyyy-MM"))
    .withColumn("rn_last", F.row_number().over(w_month))
)

monthly_agg = (
    daily
    .groupBy("year_month")
    .agg(
        F.round(F.avg("brent_reconciled_close"), 2).alias("avg_brent"),
        F.round(F.avg("usdinr_close"), 4).alias("avg_usdinr"),
        F.count("*").alias("trading_days")
    )
)

last_usdinr = (
    daily.filter(F.col("rn_last") == 1)
    .select("year_month", F.col("usdinr_close").alias("usdinr_month_end"))
)

monthly = monthly_agg.join(last_usdinr, on="year_month", how="inner")

w_ym = Window.orderBy("year_month")
monthly = monthly.withColumn(
    "prev_month_brent", F.lag("avg_brent", 1).over(w_ym)
).withColumn(
    "prev_month_usdinr_end", F.lag("usdinr_month_end", 1).over(w_ym)
)

# KPI 1: Brent MoM change and CAD impact
monthly = monthly.withColumn(
    "brent_mom_change",
    F.round(F.col("avg_brent") - F.col("prev_month_brent"), 2)
).withColumn(
    "cad_impact_pct_gdp",
    F.round(F.col("brent_mom_change") / 10 * 0.5, 4)
)

# KPI 2: USDINR monthly change %
monthly = monthly.withColumn(
    "usdinr_monthly_change_pct",
    F.round(
        (F.col("usdinr_month_end") / F.col("prev_month_usdinr_end") - 1) * 100,
        4
    )
)

# ════════════════════════════════════════════════════════════════════
# STEP 2: Conflict month flag
# ════════════════════════════════════════════════════════════════════
hc_event_months = (
    event_dim
    .filter(F.col("severity").isin("HIGH", "CRITICAL"))
    .withColumn("event_ym", F.date_format("event_date", "yyyy-MM"))
    .select("event_ym")
    .distinct()
    .withColumn("is_conflict_month", F.lit(True))
)

monthly = monthly.join(
    hc_event_months,
    monthly["year_month"] == hc_event_months["event_ym"],
    "left"
).drop("event_ym").fillna({"is_conflict_month": False})

# ════════════════════════════════════════════════════════════════════
# STEP 3: CPI data integration (use try_cast for malformed '.' values)
# ════════════════════════════════════════════════════════════════════
cpi = (
    cpi_raw
    .withColumn("cpi_value", F.expr("try_cast(value as double)"))
    .filter(F.col("cpi_value").isNotNull())
    .withColumn("cpi_ym", F.date_format("record_date", "yyyy-MM"))
    .select("cpi_ym", "cpi_value")
    .filter(F.col("cpi_ym") >= "2023-12")
    .orderBy("cpi_ym")
)

w_cpi = Window.orderBy("cpi_ym")
cpi = cpi.withColumn(
    "cpi_mom_change",
    F.round(
        (F.col("cpi_value") / F.lag("cpi_value", 1).over(w_cpi) - 1) * 100,
        4
    )
)

# Join CPI for same month
monthly = monthly.join(
    cpi.select(
        F.col("cpi_ym").alias("cpi_month_curr"),
        F.col("cpi_value").alias("cpi_value_curr"),
        F.col("cpi_mom_change").alias("cpi_mom_change_curr")
    ),
    monthly["year_month"] == F.col("cpi_month_curr"),
    "left"
).drop("cpi_month_curr")

# Next-month CPI change (1-month lagged response to this month's Brent)
monthly = monthly.join(
    cpi.select(
        F.col("cpi_ym"),
        F.col("cpi_mom_change").alias("next_month_cpi_change")
    ),
    F.add_months(F.to_date(F.concat(monthly["year_month"], F.lit("-01")), "yyyy-MM-dd"), 1)
    == F.to_date(F.concat(F.col("cpi_ym"), F.lit("-01")), "yyyy-MM-dd"),
    "left"
).drop("cpi_ym")

# ════════════════════════════════════════════════════════════════════
# WRITE GOLD TABLE
# ════════════════════════════════════════════════════════════════════
gold_rupee = monthly.select(
    "year_month", "trading_days",
    "avg_brent", "brent_mom_change", "cad_impact_pct_gdp",
    "avg_usdinr", "usdinr_month_end", "usdinr_monthly_change_pct",
    "is_conflict_month",
    "cpi_value_curr", "cpi_mom_change_curr", "next_month_cpi_change"
).orderBy("year_month")

gold_rupee.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "iran_israel_capstone_project.gold.gold_rupee_inflation_channel"
)
print(f"✅ gold_rupee_inflation_channel written: {gold_rupee.count()} rows")
display(gold_rupee)

# ════════════════════════════════════════════════════════════════════
# KPI SUMMARIES
# ════════════════════════════════════════════════════════════════════
print("\n── KPI 1: $10 Brent Impact on CAD ──")
print("   Rule: Every $10/bbl rise in monthly Brent → ~0.5% GDP CAD widening")
max_brent_rise = gold_rupee.agg(F.max("brent_mom_change")).collect()[0][0]
print(f"   Largest monthly Brent rise: ${max_brent_rise}/bbl")
if max_brent_rise:
    print(f"   Estimated CAD impact: {round(max_brent_rise / 10 * 0.5, 3)}% of GDP")

print("\n── KPI 2: Rupee Depreciation — Conflict vs Non-Conflict ──")
valid = gold_rupee.filter(F.col("usdinr_monthly_change_pct").isNotNull())
conflict_avg = valid.filter(F.col("is_conflict_month")).agg(
    F.round(F.avg("usdinr_monthly_change_pct"), 4).alias("avg")
).collect()[0]["avg"]
non_conflict_avg = valid.filter(~F.col("is_conflict_month")).agg(
    F.round(F.avg("usdinr_monthly_change_pct"), 4).alias("avg")
).collect()[0]["avg"]
diff_bps = round((conflict_avg - non_conflict_avg) * 100, 1)
print(f"   Conflict months avg:     {conflict_avg}% monthly change")
print(f"   Non-conflict months avg: {non_conflict_avg}% monthly change")
print(f"   Difference: {diff_bps} basis points")

print("\n── KPI 3: Brent–CPI Lagged Correlation ──")
corr_data = gold_rupee.filter(
    F.col("avg_brent").isNotNull() & F.col("next_month_cpi_change").isNotNull()
)
n_pairs = corr_data.count()
r_value = corr_data.stat.corr("avg_brent", "next_month_cpi_change")
print(f"   Data pairs: {n_pairs} months")
print(f"   Pearson r (avg_brent[t] vs cpi_change[t+1]): {round(r_value, 4)}")
print(f"   Direction: {'Positive ✓ (higher Brent → higher next-month CPI)' if r_value > 0 else 'Negative (unexpected)'}")
print(f"   Strength: {'Strong' if abs(r_value) > 0.5 else 'Moderate' if abs(r_value) > 0.3 else 'Weak'}")